# SIMP Topology Optimization using Gridap in Julia

UQLID Lab at NAU, 2023 

Rahaman Lab at IIT Bhubaneswar 

In the following code: 

$p$ = densities 

$pf$ = filtered densities

$pth$ = thresholded densities 

### Load required packages 

In [1]:
# FEA 
using  Gridap
using  Gridap.Geometry
using  Gridap.Fields
using  Gridap.TensorValues 
using  Gridap.CellData 
using GridapTopOpt
# Meshing 
using  Gmsh 
using  GridapGmsh

using  LinearAlgebra 

# Gardient calculation 
using  ChainRulesCore, Zygote
import ChainRulesCore: rrule 

# For MMA 
using  NLopt 

# For plotting 
#using CairoMakie, GridapMakie 

### Constant parameters 
Material properties, such as modulus of elasticity and Poisson's ratio. 
The SIMP penalty is generally assumed as 3~5. 

### Create the mesh or load already created mesh file using gmsh 
A GUI is also available to use gmsh 

In [2]:
const L  = 30#e-2          
const B  = L/3#e-2          
const H  = B/5#e-2          
nx = 120
ny = 40
nz = 4
domain = (0,L,0,B,0,H)
model = CartesianDiscreteModel(domain,(nx,ny,nz))
f_Γ_D1(x) = (x[2] ≈ B) && (L - 1*L/nx <= x[1] <= L) && (H/2 - H/nz <= x[3] <= H/2 + H/nz)
update_labels!(1,model,f_Γ_D1,"SpringBoundary")

In [3]:
ϕ  = 1000.0                 # applied voltage (V)
ks = 1e7#1e6               # spring stiffness at output DOF (N/m)
volfrac  = 0.4

penal_uu = 3             # stiffness penalisation, C_mat
penal_uφ = 4             # coupling penalisation, e_mat  
penal_P  = 1             # polarisation penalisation, pol
penal_φφ = 4             # dielectric permittivity, K_mat
penal = 3         

3

In [4]:
const E = 83161
const ν = 0.3589

const G = E/(2*(1+ν)) #
const l = L/40 # Modified
const N = 0.0

const lt = l                      # Torsional length scale
const χ = 1.0                     # Polar ratio

const λ = 2*G*ν/(1 -2*ν)
const κ = 2*G*N^2/(1-N^2)
const μ = G*(1-2*(N^2))/(1-N^2)
const γ = 4*G*l^2

const β1 = 2*G*(lt^2 -2*l^2)
const α1 = 2*G*lt^2*((1/χ) -1)


function voigt2tensor4(A::Array{M,2}) where M
  if isequal(size(A),(3,3))
    return SymFourthOrderTensorValue(A[1,1], A[3,1], A[2,1],
                                     A[1,3], A[3,3], A[2,3],
                                     A[1,2], A[3,2], A[2,2])
  elseif isequal(size(A),(6,6))
    return SymFourthOrderTensorValue(A[1,1], A[6,1], A[5,1], A[2,1], A[4,1], A[3,1],
                                     A[1,6], A[6,6], A[5,6], A[2,6], A[4,6], A[3,6],
                                     A[1,5], A[6,5], A[5,5], A[2,5], A[4,5], A[3,5],
                                     A[1,2], A[6,2], A[5,2], A[2,2], A[4,2], A[3,2],
                                     A[1,4], A[6,4], A[5,4], A[2,4], A[4,4], A[3,4],
                                     A[1,3], A[6,3], A[5,3], A[2,3], A[4,3], A[3,3])
    end
end
function voigt2tensor3(A::Array{M,2}) where M
  if isequal(size(A),(2,3))
    return ThirdOrderTensorValue(A[1,1], A[2,1], A[1,3], A[2,3], A[1,3], A[2,3], A[1,2], A[2,2])
  elseif isequal(size(A),(3,6))
    return ThirdOrderTensorValue(
      A[1,1], A[2,1], A[3,1], A[1,6], A[2,6], A[3,6], A[1,5], A[2,5], A[3,5],
      A[1,6], A[2,6], A[3,6], A[1,2], A[2,2], A[3,2], A[1,4], A[2,4], A[3,4],
      A[1,5], A[2,5], A[3,5], A[1,4], A[2,4], A[3,4], A[1,3], A[2,3], A[3,3])
  end
end

function voigt2tensor2(A::Array{M,2}) where M
  return TensorValue(A)
end

ε_0 = 8.854e-12; ## for m and mm

# C_Voigt = [C11  C12  C13  0.0  0.0  0.0;
#            C12  C11  C13  0.0  0.0  0.0;
#            C13  C13  C33  0.0  0.0  0.0;
#            0.0  0.0  0.0  C44  0.0  0.0;
#            0.0  0.0  0.0  0.0  C44  0.0;
#            0.0  0.0  0.0  0.0  0.0  C66]

e_Voigt = [0.0   0.0   0.0   0.0   12.71   0.0;
           0.0   0.0   0.0  12.71  0.0     0.0;
           -5.20  -5.20 15.08 0.0    0.0     0.0]*1e-3 

K_Voigt = [762.5*ε_0     0          0; 
                0       762.5*ε_0     0;
                0          0     663.2*ε_0]

# C_mat = voigt2tensor4(C_Voigt);
e_mat = voigt2tensor3(e_Voigt);
K_mat = voigt2tensor2(K_Voigt);

## add the boundary condition tommorow trough tags

In [5]:
path = "./NewFinal3DActuatorSinglePointL$(L)_B$(B)_H$(H)/ϕ_$(ϕ)/ks$(ks)/nx$(nx)_ny$(ny)_nz$(nz)/N$(N)_l$(l)/"                                # Output path
mkpath(path) 
cd("NewFinal3DActuatorSinglePointL$(L)_B$(B)_H$(H)/ϕ_$(ϕ)/ks$(ks)/nx$(nx)_ny$(ny)_nz$(nz)/N$(N)_l$(l)/")

In [6]:
writevtk(model,"3DActuator")

4-element Vector{Vector{String}}:
 ["3DActuator_0.vtu"]
 ["3DActuator_1.vtu"]
 ["3DActuator_2.vtu"]
 ["3DActuator_3.vtu"]

### Constitutive matrix (1 for plane stress, 2 for plane strain) 
Elasticity tensor, constitutive relation, and SIMP penalty relation are formulated here. 

In [7]:
I2 = SymTensorValue{3,Float64}(1.0, 0.0, 0.0, 1.0, 0.0, 1.0)
function σ_Bmod(ε)
    σM = (λ*tr(ε)*one(ε) + (2*μ + κ)*(ε))
    return σM
end
# function σ_Bmod(ε)
#     σM = C_mat ⊙ ε
#     return σM
# end

function σ_piezo(∇)
    σ = ∇ ⋅ e_mat 
    return  σ
end 

function D_piezo(∇)
    σ = K_mat ⋅ ∇
    return  σ
end

function D_elas(ε)
    σ = e_mat ⋅² ε
    return  σ
end

E_Matrix = zeros(3,3,3)
E_Matrix[1,2,3]=1.0
E_Matrix[2,3,1]=1.0
E_Matrix[3,1,2]=1.0
E_Matrix[1,3,2]=-1.0
E_Matrix[3,2,1]=-1.0
E_Matrix[2,1,3]=-1.0
E_Tensor = ThirdOrderTensorValue(E_Matrix ...)

function ε_Skw(∇,θ)
    ∇ᵀ = transpose(∇)
    w = (0.5*(∇ᵀ - ∇)) - (E_Tensor ⋅ θ)
    return w
end

function σ_Cmod(ϵ_skew)
    σM = κ*ϵ_skew
    return σM
end


function M_mod(∇)
    M = α1*(∇ ⊙ I2)*I2 + β1*∇ + γ*∇
    return M
end

function Skw(u,θ)
    ∇ᵀ = transpose(∇(u))
    w = (0.5*(∇ᵀ - ∇(u)) - (E_Tensor ⋅ θ))
    return w
end

function Em(p)
    Em = p ^ penal
    return Em
end 

Em (generic function with 1 method)

In [8]:
const Emin = 1e-9
const eMin = 1e-9
const E0   = 1.0
const e0   = 1.0

function Em_uu(x)
    return Emin + (x ^ penal_uu) * (E0 - Emin)
end

function Em_uφ(x, P)
    return (eMin + (x ^ penal_uφ) * (e0 - eMin)) * (2*P - 1)^penal_P
end

function Em_φφ(x)
    return (eMin + (x ^ penal_φφ) * (e0 - eMin))
end
## use Em_uu instead of Em_φφ.
function Em(p)
    Em = p ^ penal_uu
    return Em
end 

Em (generic function with 1 method)

### Finite Element Analysis 

In [9]:
order = 1
degree = 2*order
Ω= Triangulation(model)
dΩ= Measure(Ω,degree) 

reffe_Disp = ReferenceFE(lagrangian ,VectorValue{3,Float64},order)
V0_Disp = TestFESpace(model,reffe_Disp;conformity =:H1,
    dirichlet_tags = ["tag_01","tag_03","tag_07","tag_05","tag_13","tag_19","tag_15","tag_17","tag_25"],
    dirichlet_masks =[(true,true,true),(true,true,true),(true,true,true),(true,true,true),(true,true,true),
                     (true,true,true),(true,true,true),(true,true,true),(true,true,true)])
U0_Disp = TrialFESpace(V0_Disp, [VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),
                     VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),
                     VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0)])

reffe_ElecPot = ReferenceFE(lagrangian,Float64,order)
V0_ElecPot  = TestFESpace(model,reffe_ElecPot;conformity=:H1,
  dirichlet_tags=["tag_05","tag_07","tag_08","tag_06","tag_15","tag_12","tag_16","tag_11","tag_22",
                  "tag_01","tag_03","tag_04","tag_02","tag_09","tag_14","tag_10","tag_13","tag_21"])    
U0_ElecPot = TrialFESpace(V0_ElecPot,[ϕ,ϕ,ϕ,ϕ,ϕ,ϕ,ϕ,ϕ,ϕ,
                                      0,0,0,0,0,0,0,0,0])
U0_ElecPotAdj = TrialFESpace(V0_ElecPot,[0,0,0,0,0,0,0,0,0,
                                         0,0,0,0,0,0,0,0,0])

reffe_theta = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
# V0_theta  = TestFESpace(model,reffe_theta;conformity=:H1)
# U0_theta = TrialFESpace(V0_theta)
V0_theta = TestFESpace(model,reffe_theta;conformity =:H1,
    dirichlet_tags = ["tag_01","tag_03","tag_07","tag_05","tag_13","tag_19","tag_15","tag_17","tag_25"],
    dirichlet_masks =[(true,true,true),(true,true,true),(true,true,true),(true,true,true),(true,true,true),
                     (true,true,true),(true,true,true),(true,true,true),(true,true,true)])
U0_theta = TrialFESpace(V0_theta, [VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),
                     VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),
                     VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0),VectorValue(0.0, 0.0,0.0)])

V0 = MultiFieldFESpace([V0_Disp,V0_ElecPot,V0_theta])
U0 = MultiFieldFESpace([U0_Disp,U0_ElecPot,U0_theta])

labels = get_face_labeling(model)
LoadTagId = get_tag_from_name(labels ,"SpringBoundary")
Γ_Load = BoundaryTriangulation(model ;tags = LoadTagId)
dΓ_Load = Measure(Γ_Load ,degree)
n_Γ_Load = get_normal_vector(Γ_Load) 
vol_Γ_Load   = sum(∫(1) * dΓ_Load)   # length of spring patch

p_reffe = ReferenceFE(lagrangian, Float64, 0)
Q = TestFESpace(Ω, p_reffe, vector_type = Vector{Float64})
P = Q
np = num_free_dofs(P) 

pf_reffe = ReferenceFE(lagrangian, Float64, 1)
Qf = TestFESpace(Ω, pf_reffe, vector_type = Vector{Float64})
Pf = Qf 

#fem_params = (;V0_Disp, U0_Disp, Q, P, Qf, Pf, np, Ω, dΩ)  #, n_Γ_Load, dΓ_Load
fem_params = (; V0_Disp, U0_Disp, V0_theta, U0_theta, V0_ElecPot, U0_ElecPot,U0_ElecPotAdj, V0, U0, Q, P, Qf, Pf, np, Ω, dΩ, n_Γ_Load, dΓ_Load,vol_Γ_Load)

function get_dof_for_node_component(V0_Disp, node_id, component)
    dof_to_node = V0_Disp.metadata.node_and_comp_to_dof
    return dof_to_node[node_id][component]   # index node first, then component
end

# Node indices (all 1-based)
i = nx + 1              # x = L   → last node in x
j = ny + 1              # y = L   → last node in y  ← changed
k = (nz ÷ 2) + 1       # z = L/2 → middle node in z

node_id = i + (j-1)*(nx+1) + (k-1)*(nx+1)*(ny+1)

coords = model.grid.node_coords
println("Node $node_id coords: ", coords[node_id])
# Should print approximately: (L, L, L/2)

DMDOF = get_dof_for_node_component(fem_params.V0_Disp, node_id, 2)  # ← component 2 for y
println("DMDOF = ", DMDOF)
DMDOF = Int(DMDOF)

e2 = VectorValue(0.0, 1.0, 0.0) 

function stepDisp2(fem_params,pth,polh,DMDOF)
        a((u,ϕ,θ),(w,ψ,v))  = ∫( ((p->Em_uu(p))∘pth) * (ε(w) ⊙ (σ_Bmod∘(ε(u))) ) + (((p,P) -> Em_uφ(p,P)) ∘ (pth, polh)) * ((∇(ϕ) ⋅ e_mat) ⊙ ∇(w)) - ((p->Em_uu(p))∘pth) * ((D_piezo∘ (∇(ϕ))) ⋅ ∇(ψ)) + (((p,P) -> Em_uφ(p,P)) ∘ (pth, polh)) * ((e_mat ⋅² ε(u)) ⋅ ∇(ψ))  + (((p -> Em_uu(p)) ∘ pth)) * ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) ) + (((p -> Em_uu(p)) ∘ pth)) * (∇(v)) ⊙ (M_mod∘(∇(θ))) - (((p -> Em_uu(p)) ∘ pth))* (v ⋅ ((E_Tensor) ⋅² (σ_Cmod∘(ε_Skw∘(∇(u),θ))))) )*dΩ + ∫(ks * (((u ⋅ e2) * (w ⋅ e2))) ) * dΓ_Load
        b((w,ψ,v)) = 0
        op    = AffineFEOperator(a, b, U0, V0)
        uh = solve(op)
        K   = get_matrix(op)
        F   = get_vector(op)
    return uh, K
end

Node 14883 coords: (30.0, 10.0, 1.0)
DMDOF = 44279


stepDisp2 (generic function with 1 method)

### Filtering and thresholding 
Helmholtz PDE-based filter is used here. 

(see: Lazarov, Boyan Stefanov, and Ole Sigmund. "Filters in topology optimization based on Helmholtz‐type differential equations." International Journal for Numerical Methods in Engineering 86.6 (2011): 765-781.) 

A smooth approximation of the Heaviside step function is used as the thresholding function. $\beta$ determines the strength of the approximation, and $\eta$ determines the location. Note that $\eta=0.5$ is standard in TO. 

In [10]:
elemsize = maximum(get_el_Δ(model))
r = 1*elemsize#(3*elemsize)/(2*sqrt(3)) #0.025           # Filter radius
β = 4                       # β∈[1,∞], threshold sharpness
η = 0.5                     # η∈[0,1], threshold center

a_f(r, u, v) = r^2 * (∇(v) ⋅ ∇(u))

function Filter(p0; r, fem_params)
    ph = FEFunction(fem_params.P, p0)
    op = AffineFEOperator(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ, ∫(v * ph)fem_params.dΩ
      end
    pfh = solve(op)
    return get_free_dof_values(pfh)
end

function Threshold(pfh; β, η)
    return  ((tanh(β * η) + tanh(β * (pfh - η))) / (tanh(β * η) + tanh(β * (1.0 - η)))) 

end 


Dptdpf(pf; β, η) = β * (1.0 - tanh(β * (pf - η))^2) / (tanh(β * η) + tanh(β * (1.0 - η))) # Gradient of thresholding function 
DEdpf_uu(pf; β, η) = penal_uu * Threshold(pf; β, η)^(penal_uu-1) * Dptdpf(pf; β, η) * (E0 - Emin)
DEdpf_uφ(pf; β, η) = penal_uφ * Threshold(pf; β, η)^(penal_uφ-1) * Dptdpf(pf; β, η) * (e0 - eMin)
DEdpf_φφ(pf; β, η) = penal_φφ * Threshold(pf; β, η)^(penal_φφ-1) * Dptdpf(pf; β, η) * (e0 - eMin)
DEdpol(P)          = penal_P  * (2*P - 1)^(penal_P-1) * 2 * (e0 - eMin)


function DAdpf(u, ϕ, θ, w, ψ, v, pfh, polh; β, η)
    return (( ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * (ε(w) ⊙ (σ_Bmod∘(ε(u))) ) 
           + ((p -> DEdpf_uφ(p; β, η)) ∘ pfh) * ((P -> (2*P - 1)^penal_P) ∘ polh) * ((∇(ϕ) ⋅ e_mat) ⊙ ∇(w)) 
           - ((p -> DEdpf_φφ(p; β, η)) ∘ pfh) * ((D_piezo∘ (∇(ϕ)))⋅ ∇(ψ)) 
           + ((p -> DEdpf_uφ(p; β, η)) ∘ pfh) * ((P -> (2*P - 1)^penal_P) ∘ polh) * ((e_mat ⋅² ε(u)) ⋅ ∇(ψ)) 
           + ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) )  
           + ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * ((∇(v)) ⊙ (M_mod∘(∇(θ))))  
           - ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * (v ⋅ ((E_Tensor) ⋅² (σ_Cmod∘(ε_Skw∘(∇(u),θ))))) ))
end

function DAdpol(u, ϕ, θ, w, ψ, v, pth, polh; β, η)
    return (( ((P -> DEdpol(P)) ∘ polh) * ((p -> p^penal_uφ) ∘ pth) * ((∇(ϕ) ⋅ e_mat) ⊙ ∇(w))  
            + ((P -> DEdpol(P)) ∘ polh) * ((p -> p^penal_uφ) ∘ pth) * ((e_mat ⋅² ε(u)) ⋅ ∇(ψ)) )) 
end

Dfdpf(u,ϕ,θ,pfh; β, η)  = 0.0

Dfdpf (generic function with 1 method)

### FEA result with the entire domain filled with density 1 elements 
For checking! 

In [11]:
p0_init   = ones(fem_params.np)
pol0_init = fill(1.0, fem_params.np)

pf_vec_chk = Filter(p0_init; r, fem_params)
pfh_chk    = FEFunction(fem_params.Pf, pf_vec_chk)
pth_chk    = (pf -> Threshold(pf; β, η)) ∘ pfh_chk

polh_chk     = FEFunction(fem_params.P, pol0_init)
polh_eff_chk = Operation(P -> (2*P - 1)^penal_P)(polh_chk)

uh_comb_chk, K_mat1 = stepDisp2(fem_params, pth_chk, polh_chk, DMDOF)
uh_u_chk, uh_ϕ_chk, uh_θ_chk = uh_comb_chk

filename1 = "SanityCheck_FullSolid_K$(replace(string(ks), "." => "_"))"
writevtk(Ω, filename1, cellfields = [
    "uh"      => uh_u_chk,
    "phi"     => uh_ϕ_chk,
    "theta"   => uh_θ_chk,
    "pth"     => pth_chk,
    "pol"     => polh_chk,
    "pol_eff" => polh_eff_chk
])
println("Sanity-check VTK written: $(filename1).vtu")

uh_vec_chk = get_free_dof_values(uh_comb_chk)
println("Full-solid U_out = ", uh_vec_chk[DMDOF], " m")

Sanity-check VTK written: SanityCheck_FullSolid_K1_0e7.vtu
Full-solid U_out = -1.5394377943157119e-7 m


In [12]:
u2_load = uh_u_chk ⋅ VectorValue(0.0, 1.0, 0.0) 

avg_u2_sanity = sum(∫(u2_load)*dΓ_Load) / sum(∫(1.0)*dΓ_Load)

println("Average u2 on dΓ_Load = ", avg_u2_sanity, " m")

Average u2 on dΓ_Load = 1.4763304537654862e-7 m


### Gradient calculation 
Using adjoint method 

In [13]:
function pf_p0(p0; r, fem_params)
    pf_vec = Filter(p0; r, fem_params)
    pf_vec
end

function Dgdp(dgdpf; r, fem_params)
    Af = assemble_matrix(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ
    end
    wvec = Af' \ dgdpf
    wh = FEFunction(fem_params.Pf, wvec)
    l_temp(dp) = ∫(wh * dp)fem_params.dΩ
    return assemble_vector(l_temp, fem_params.P)
end 

function gf_p(vars::Vector, grad::Vector; r, β, η, fem_params)
    np  = fem_params.np
    p0  = vars[1:np]
    pol = vars[np+1:end]
    
    pf_vec = Filter(p0; r, fem_params)
    pfh    = FEFunction(fem_params.Pf, pf_vec)
    pth    = (pf -> Threshold(pf; β, η)) ∘ pfh
    
    polh = FEFunction(fem_params.P, pol)

    uh_comb, A_mat = stepDisp2(fem_params, pth, polh, DMDOF)
    uh_u, uh_ϕ, uh_θ = uh_comb

    U_out  = sum(∫((uh_u ⋅ e2)/vol_Γ_Load) * fem_params.dΓ_Load)
    gf_val = -U_out * 1e6

    # ── Gradient ───────────────────────────────────────────────────────────
    if length(grad) > 0

        dfduFn(w) = ∫((-1e6 * (w ⋅ e2))/vol_Γ_Load) * fem_params.dΓ_Load
        dfduVec   = assemble_vector(dfduFn, fem_params.U0_Disp)

        dfdϕVec = zeros(num_free_dofs(fem_params.U0_ElecPotAdj))
        
        dfdθVec = zeros(num_free_dofs(fem_params.U0_theta))

        dfduϕθVec = vcat(-dfduVec, -dfdϕVec, -dfdθVec)

        wvec = A_mat' \ dfduϕθVec

        n_u   = num_free_dofs(fem_params.U0_Disp)
        n_ϕ   = num_free_dofs(fem_params.U0_ElecPotAdj)     
        wUvec = wvec[1:n_u]
        wϕvec = wvec[n_u+1:n_u+n_ϕ]
        wθvec = wvec[n_u+n_ϕ+1:end]

        λh_u = FEFunction(fem_params.U0_Disp, wUvec)
        λh_ϕ = FEFunction(fem_params.U0_ElecPotAdj, wϕvec) #λh_ϕ = FEFunction(fem_params.U0_ElecPot, wϕvec)
        λh_θ = FEFunction(fem_params.U0_theta, wθvec)

        # ── Density sensitivity ───────────────────────────────────────────────
        wtimsdgdp_density(dp) =
            ∫(DAdpf(uh_u, uh_ϕ, uh_θ, λh_u, λh_ϕ, λh_θ, pfh, polh; β, η) * dp ) * fem_params.dΩ

        dgfdpf       = assemble_vector(wtimsdgdp_density, fem_params.Pf)
        grad_density = Dgdp(dgfdpf; r, fem_params)

        # ── Polarisation sensitivity ──────────────────────────────────────────
        wtimsdgdp_pol(dpol) =
            ∫( DAdpol(uh_u, uh_ϕ, uh_θ, λh_u, λh_ϕ, λh_θ, pth, polh; β, η) * dpol ) * fem_params.dΩ      
        grad_pol = assemble_vector(wtimsdgdp_pol, fem_params.P)

        grad[:] = vcat(collect(grad_density), collect(grad_pol))
    end
        # ── Gradient VTK output ───────────────────────────────────────────────
    grad_density_h = FEFunction(fem_params.P, collect(grad_density))
    grad_pol_h     = FEFunction(fem_params.P, collect(grad_pol))

    grad_density_pf_h = FEFunction(fem_params.Pf, collect(dgfdpf))

    fname_grad = "GradResults_K$(replace(string(ks), "." => "_"))"
    writevtk(fem_params.Ω, fname_grad, cellfields = [
        "grad_density_raw"     => grad_density_pf_h,   
        "grad_density_filtered"=> grad_density_h,    
        "grad_pol"             => grad_pol_h,
    ])
    # ── 5. VTK output ─────────────────────────────────────────────────────────
    polh_eff = Operation(P -> (2*P - 1)^penal_P)(polh)
    fname    = "OptResults_K$(replace(string(ks), "." => "_"))"
    writevtk(fem_params.Ω, fname, cellfields = [
        "uh"       => uh_u,
        "phi"      => uh_ϕ,
        "theta"    => uh_θ,
        "pth"      => pth,
        "polh"     => polh,
        "polh_eff" => polh_eff
    ])

    open("gvaluemma.txt", "a") do io
        write(io, "$gf_val\n")
    end

    return gf_val
end

gf_p (generic function with 1 method)

### Volume constraint calculation 
Volume should be estimated with filtered densities and not with thresholded densities. 

In [14]:
function  project(q,model ,dΩ,order)
    reffe = ReferenceFE(lagrangian ,Float64 ,order)
    V = FESpace(model ,reffe ,conformity =:L2)
    a(u,v) =∫(u*v)*dΩ
    b(v) =∫(v*q)*dΩ
    op = AffineFEOperator(a,b,V,V)
    qh = solve(op)
    return  qh
end

project (generic function with 1 method)

In [15]:
open("volume_history.txt", "w") do io
    println(io, "iter, volume_constraint")
end

In [16]:
# Iteration counter (use a Ref so it's mutable inside the closure)
iter_count = Ref(0)
dv = get_array(∫(1)fem_params.dΩ)
function cf_p(vars::Vector, gradc::Vector; r, β, η, fem_params)
    np  = fem_params.np
    p0  = vars[1:np]
    pf_vec = Filter(p0; r, fem_params)
    pfh    = FEFunction(fem_params.Pf, pf_vec)
    if length(gradc) > 0
        gradc[:] = vcat(dv, zeros(np))
    end

    val = sum(∫(pfh) * fem_params.dΩ) - volfrac * sum(dv)

    # --- log to file ---
    iter_count[] += 1
    open("volume_history.txt", "a") do io
        println(io, "$(iter_count[]), $(val)")
    end

    return val
end

cf_p (generic function with 1 method)

### MMA to optimize 
Svanberg, Krister. "The method of moving asymptotes—a new method for structural optimization." International journal for numerical methods in engineering 24.2 (1987): 359-373. 

In [17]:
function gf_p_optimize(vars_init::Vector; r, β, η, TOL=1e-4, MAX_ITER, fem_params)
    np = fem_params.np

    opt = Opt(:LD_MMA, 2 * np)

    # Bounds: density ∈ [0.001, 1],  polarisation ∈ [0, 1]
    opt.lower_bounds = vcat(fill(0.001, np), fill(0.0, np))
    opt.upper_bounds = vcat(fill(1.0,   np), fill(1.0, np))

    opt.xtol_rel  = TOL
    opt.maxeval   = MAX_ITER

    opt.min_objective = (vars, grad) -> gf_p(vars, grad; r, β, η, fem_params)

    inequality_constraint!(opt,
        (vars, gradc) -> cf_p(vars, gradc; r, β, η, fem_params), 1e-8)

    (g_opt, vars_opt, ret) = optimize(opt, vars_init)

    p_opt   = vars_opt[1:np]
    pol_opt = vars_opt[np+1:end]

    @show numevals = opt.numevals
    println("got $(g_opt) after $numevals iterations (returned $ret)")

    return g_opt, vars_opt, ret
end

gf_p_optimize (generic function with 1 method)

### Run the MMA 

In [18]:
np_chk = fem_params.np

p0_fd   = fill(volfrac, np_chk)
pol0_fd = fill(1.0, np_chk)
vars0   = vcat(p0_fd, pol0_fd)

δ = 1e-3

# Density perturbation at element 30
δp_disp        = zeros(2 * np_chk)
δp_disp[30]    = δ

# Polarisation perturbation at element 30
δp_pol         = zeros(2 * np_chk)
δp_pol[np_chk + 30] = δ

grad  = zeros(2 * np_chk)
gradm = zeros(2 * np_chk)

g0      = gf_p(vars0,            grad;  r, β, η, fem_params)
g1_disp = gf_p(vars0 + δp_disp, gradm; r, β, η, fem_params)
g1_pol  = gf_p(vars0 + δp_pol,  gradm; r, β, η, fem_params)

println("==========================================")
println("FD CHECK — DENSITY")
println("==========================================")
println("FD approximation  : $(g1_disp - g0)")
println("Adjoint prediction: $(grad' * δp_disp)")
println("Relative error    : $((g1_disp - g0 - grad'*δp_disp) / (g1_disp - g0))")
println("==========================================")
println("FD CHECK — POLARISATION")
println("==========================================")
println("FD approximation  : $(g1_pol - g0)")
println("Adjoint prediction: $(grad' * δp_pol)")
println("Relative error    : $((g1_pol - g0 - grad'*δp_pol) / (g1_pol - g0))")
println("==========================================")

FD CHECK — DENSITY
FD approximation  : -2.6661417580186447e-8
Adjoint prediction: -2.665987688278063e-8
Relative error    : 5.778752765797334e-5
FD CHECK — POLARISATION
FD approximation  : -1.0155607643030917e-8
Adjoint prediction: -1.0155807045794047e-8
Relative error    : -1.963474467887117e-5


### Run the MMA 

In [19]:
np_run = fem_params.np

p0_run   = fill(volfrac, np_run)
pol0_run = fill(1.0, np_run)

vars0_run = vcat(p0_run, pol0_run)

TOL      = 1e-10
MAX_ITER = 500

g_opt, vars_opt, ret = gf_p_optimize(vars0_run; r, β, η, TOL, MAX_ITER, fem_params)

p_opt   = vars_opt[1:np_run]
pol_opt = vars_opt[np_run+1:end]

numevals = opt.numevals = 500
got -1.783447582078867 after 500 iterations (returned MAXEVAL_REACHED)


19200-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 1.0
 1.0

### Postprocessing 
Redefine the thresholding function with a sharp increase (i.e., increase β) to get a crisp boundary 

In [20]:
β_post = 64    # sharper threshold for crisp boundary visualisation

pf_vec_opt = Filter(p_opt; r, fem_params)
pfh_opt    = FEFunction(fem_params.Pf, pf_vec_opt)
pth_opt    = (pf -> Threshold(pf; β=β_post, η)) ∘ pfh_opt

polh_opt     = FEFunction(fem_params.P, pol_opt)
polh_eff_opt = Operation(P -> (2*P - 1)^penal_P)(polh_opt)

uh_comb_opt, _ = stepDisp2(fem_params, pth_opt, polh_opt, DMDOF)
uh_u_opt, uh_ϕ_opt, uh_θ_opt = uh_comb_opt

fname_final = "FinalResult_K$(replace(string(ks), "." => "_"))"
writevtk(Ω, fname_final, cellfields = [
    "uh"       => uh_u_opt,
    "phi"      => uh_ϕ_opt,
    "theta"    => uh_θ_opt,
    "pth"      => pth_opt,
    "pol_opt"  => polh_opt,
    "pol_eff"  => polh_eff_opt
])
println("Final VTK written: $(fname_final).vtu")
println("Final U_out = ", get_free_dof_values(uh_comb_opt)[DMDOF], " m")

Final VTK written: FinalResult_K1_0e7.vtu
Final U_out = -6.1592898619328e-7 m


In [21]:
u2_load_opt = uh_u_opt ⋅ VectorValue(0.0, 1.0, 0.0) 

avg_u2_opt = sum(∫(u2_load_opt)*dΓ_Load) / sum(∫(1.0)*dΓ_Load)

println("Average u2 on dΓ_Load after optimization = ", avg_u2_opt, " m")

Average u2 on dΓ_Load after optimization = 2.434265921935485e-6 m


In [22]:
open("u2_load_summary.txt", "a") do io
    write(io, "ks = $ks:\n")
    write(io, "  avg_u2 (sanity)     = $avg_u2_sanity m\n")
    write(io, "  avg_u2 (opt)        = $avg_u2_opt m\n")
end

47